# Objective

Validate whether generated answer is:

- Grounded in retrieved documents
- Relevant to user query
- Not hallucinated
- Supported by evidence
- Safe to send forward

### Import Libraries

In [1]:
import os
import re
import json
import pandas as pd

from dotenv import load_dotenv

from langchain_groq import ChatGroq

### Load Environment Variables

In [2]:
load_dotenv()

GROQ_API_KEY = os.getenv("GROQ_API_KEY")

### Load Validation LLM

In [3]:
validator_llm = ChatGroq(
    groq_api_key=GROQ_API_KEY,
    model_name="llama-3.3-70b-versatile",
    temperature=0
)

print("Validation LLM Loaded")

Validation LLM Loaded


### Sample Inputs

In [4]:
user_query = "How can I reset my debit card PIN?"

retrieved_context = """
You can reset your debit card PIN through
internet banking, mobile banking,
ATM PIN generation service,
or by visiting the branch.
"""

generated_answer = """
You can reset your debit card PIN
through internet banking,
mobile banking,
ATM PIN generation,
or by visiting your bank branch.
"""

### Groundedness Validation Prompt

In [7]:
grounding_prompt = """
You are a Banking AI Validator.

User Question:
{query}

Retrieved Context:
{context}

Generated Answer:
{answer}

Evaluate:

1. Is the answer fully supported by context?
2. Any unsupported claims?
3. Any missing evidence?

Return JSON:

{{
    "grounded": true/false,
    "confidence": 0-100,
    "reason": "short explanation"
}}
"""

#### Groundedness Function

In [8]:
def validate_groundedness(
    query,
    context,
    answer
):

    prompt = grounding_prompt.format(
        query=query,
        context=context,
        answer=answer
    )

    response = validator_llm.invoke(prompt)

    return response.content

#### Run Groundedness Check

In [9]:
grounding_result = validate_groundedness(
    user_query,
    retrieved_context,
    generated_answer
)

print(grounding_result)

{
    "grounded": true,
    "confidence": 100,
    "reason": "The answer is fully supported by the context, with all options for resetting the debit card PIN correctly listed."
}


### Hallucination Detection Prompt

In [10]:
hallucination_prompt = """
You are a Banking Hallucination Detector.

Question:
{query}

Retrieved Context:
{context}

Answer:
{answer}

Identify:

1. Unsupported statements
2. Fabricated facts
3. Invented banking policies
4. Incorrect RBI regulations

Return JSON:

{{
    "hallucination": true/false,
    "confidence": 0-100,
    "reason": "short explanation"
}}
"""

#### Hallucination Function

In [11]:
def detect_hallucination(
    query,
    context,
    answer
):

    prompt = hallucination_prompt.format(
        query=query,
        context=context,
        answer=answer
    )

    response = validator_llm.invoke(prompt)

    return response.content

#### Run Hallucination Detection

In [12]:
hallucination_result = detect_hallucination(
    user_query,
    retrieved_context,
    generated_answer
)

print(hallucination_result)

To determine if the provided answer contains any hallucinations, let's analyze it against the retrieved context.

The answer states: "You can reset your debit card PIN through internet banking, mobile banking, ATM PIN generation, or by visiting your bank branch."

The retrieved context states: "You can reset your debit card PIN through internet banking, mobile banking, ATM PIN generation service, or by visiting the branch."

Comparing the two:

1. **Unsupported statements**: None found.
2. **Fabricated facts**: The term "ATM PIN generation service" in the context is slightly different from "ATM PIN generation" in the answer, but this does not significantly alter the meaning or introduce a fabricated fact.
3. **Invented banking policies**: None found.
4. **Incorrect RBI regulations**: The answer does not mention any specific RBI regulations, so there's nothing to identify as incorrect in this regard.

However, there is a minor deviation in wording ("ATM PIN generation service" vs. "ATM 

### Evidence Coverage Check

In [13]:
coverage_prompt = """
You are a Banking Evidence Validator.

Question:
{query}

Context:
{context}

Answer:
{answer}

Rate evidence coverage.

Return JSON:

{{
    "coverage_score": 0-100,
    "reason": "short explanation"
}}
"""

#### Coverage Function

In [14]:
def check_evidence_coverage(
    query,
    context,
    answer
):

    prompt = coverage_prompt.format(
        query=query,
        context=context,
        answer=answer
    )

    response = validator_llm.invoke(prompt)

    return response.content

#### Run Coverage Check

In [15]:
coverage_result = check_evidence_coverage(
    user_query,
    retrieved_context,
    generated_answer
)

print(coverage_result)

{
    "coverage_score": 100,
    "reason": "The answer fully covers all the available options for resetting a debit card PIN as provided in the context."
}


### Response Relevance Validation

In [16]:
relevance_prompt = """
You are a Banking Relevance Validator.

Question:
{query}

Answer:
{answer}

Evaluate:

1. Relevance
2. Completeness
3. Directness

Return JSON:

{{
    "relevant": true/false,
    "score": 0-100,
    "reason": "short explanation"
}}
"""

#### Relevance Function

In [17]:
def validate_relevance(
    query,
    answer
):

    prompt = relevance_prompt.format(
        query=query,
        answer=answer
    )

    response = validator_llm.invoke(prompt)

    return response.content

#### Run Relevance Check

In [18]:
relevance_result = validate_relevance(
    user_query,
    generated_answer
)

print(relevance_result)

```json
{
    "relevant": true,
    "score": 90,
    "reason": "The answer provides multiple relevant options for resetting a debit card PIN, covering both digital and physical channels, but could be improved with more specific instructions or details for each option."
}
```


### Unified Validation Pipeline

In [19]:
def validate_response(
    query,
    context,
    answer
):

    grounding = validate_groundedness(
        query,
        context,
        answer
    )

    hallucination = detect_hallucination(
        query,
        context,
        answer
    )

    coverage = check_evidence_coverage(
        query,
        context,
        answer
    )

    relevance = validate_relevance(
        query,
        answer
    )

    return {
        "grounding": grounding,
        "hallucination": hallucination,
        "coverage": coverage,
        "relevance": relevance
    }

#### Execute Full Validation

In [20]:
validation_report = validate_response(
    user_query,
    retrieved_context,
    generated_answer
)

validation_report

{'grounding': '{\n    "grounded": true,\n    "confidence": 100,\n    "reason": "The answer is fully supported by the context, with all options for resetting the debit card PIN correctly listed."\n}',
 'hallucination': 'To determine if the provided answer contains any hallucinations, let\'s analyze it against the retrieved context.\n\nThe answer states that you can reset your debit card PIN through:\n1. Internet banking\n2. Mobile banking\n3. ATM PIN generation\n4. Visiting your bank branch\n\nComparing this with the retrieved context, which lists the same methods:\n1. Internet banking\n2. Mobile banking\n3. ATM PIN generation service\n4. Visiting the branch\n\nThe answer closely matches the context, with a minor variation in wording for "ATM PIN generation service" being shortened to "ATM PIN generation" and "visiting the branch" being more specifically stated as "visiting your bank branch." These variations do not significantly alter the meaning or introduce any unsupported, fabricate

### Convert Validation Report to DataFrame

In [21]:
report_df = pd.DataFrame(
    validation_report.items(),
    columns=["Check", "Result"]
)

report_df

,Check,Result
0,grounding,"{\n ""grounded"": true,\n ""confidence"": 10..."
1,hallucination,To determine if the provided answer contains a...
2,coverage,"{\n ""coverage_score"": 100,\n ""reason"": ""..."
3,relevance,"```json\n{\n ""relevant"": true,\n ""score""..."


### Save Validation Results

In [22]:
os.makedirs(
    "../data/output",
    exist_ok=True
)

report_df.to_csv(
    "../data/output/response_validation_report.csv",
    index=False
)

print("Validation Report Saved")

Validation Report Saved


### Validation Pass/Fail Logic

In [23]:
def validation_gate(
    grounded,
    hallucination,
    coverage_score,
    relevance_score
):

    if not grounded:
        return "FAIL"

    if hallucination:
        return "FAIL"

    if coverage_score < 70:
        return "FAIL"

    if relevance_score < 70:
        return "FAIL"

    return "PASS"

## Key Insights

### Purpose

This notebook validates RAG-generated responses before they are shown to users.

### Validation Checks

1. Groundedness Validation
   - Ensures answer is supported by retrieved context.

2. Hallucination Detection
   - Detects fabricated or unsupported information.

3. Evidence Coverage
   - Measures how much of the answer is backed by source documents.

4. Relevance Validation
   - Ensures the answer addresses the user’s question directly.

### Output

The notebook generates a structured validation report that can be consumed by downstream trust-scoring and governance layers.

### Enterprise Importance

For banking-domain AI systems, response validation reduces the risk of:
- Incorrect financial advice
- Unsupported claims
- Regulatory violations
- Hallucinated banking policies

This notebook forms the first trust layer after RAG generation and before guardrails, multi-LLM judging, and trust-score calculation.